# Lesson 03 — SQLAlchemy ORM Models

**Run this in Google Colab** — every student gets the same environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mpalomera/learning-sql/blob/master/lessons/06-sqlalchemy-orm-migrations/03_sqlalchemy_models.ipynb)

---

## Step 1: Install dependencies

In [ ]:
!pip install sqlalchemy oracledb -q

## Step 2: Define ORM models

In [61]:
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, func
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = "teams"
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users = relationship("User", back_populates="team")
      # NEW COLUMNS:
    priority = Column(String(20), default='medium')
    due_date = Column(DateTime)
    tags = Column(String(500))

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"

class User(Base):
    __tablename__ = "users"
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey("teams.id"))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team  = relationship("Team", back_populates="users")
    tasks = relationship("Task", back_populates="assignee")
    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

class Task(Base):
    __tablename__ = "tasks"
    id           = Column(Integer, primary_key=True)
    title        = Column(String(200), nullable=False)
    description  = Column(String(1000))
    status       = Column(String(20), default="open")
    assigned_to  = Column(Integer, ForeignKey("users.id"))
    created_at   = Column(DateTime, server_default=func.current_timestamp())
    updated_at   = Column(DateTime, onupdate=func.current_timestamp())
    assignee = relationship("User", back_populates="tasks")
    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

print("✅ Models defined: Team, User, Task")

✅ Models defined: Team, User, Task


## Step 3: Connect and query with ORM

In [ ]:
# ============================================================
# CONFIGURATION — replace with your FreeSQL credentials
# ============================================================
USERNAME = ""
PASSWORD = ""   # <-- REPLACE THIS
DSN = "tcps://db.freesql.com:2484/23ai_34ui2"

engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

with Session(engine) as session:
    print("🏢 Teams:")
    for team in session.query(Team).all():
        print(f"   {team}")
        for user in team.users:
            print(f"      -> {user.full_name} ({user.username})")

    print("\n📝 Tasks with assignees:")
    for task in session.query(Task).all():
        assignee = task.assignee.full_name if task.assignee else "Unassigned"
        print(f"   {task.title} → {assignee}")

print("\n✅ ORM models working! Relationships navigate automatically.")

🏢 Teams:
   <Team(id=1, name='Engineering')>
      -> Alice Smith (alice_dev)
      -> Bob Jones (bob_dev)
   <Team(id=2, name='Product')>
      -> Carol White (carol_pm)

📝 Tasks with assignees:
   Fix login bug → Alice Smith
   Design new dashboard → Carol White
   Update dependencies → Bob Jones

✅ ORM models working! Relationships navigate automatically.


# Lesson 03 — Alembic Migrations (Pure Python)


In [ ]:
!pip install sqlalchemy oracledb alembic -q

In [ ]:
import os
from alembic.config import Config
from alembic import command

# Create a minimal alembic.ini in memory
alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Initialize the migration directory
!mkdir -p /content/alembic/versions

# Write a minimal env.py
env_py = '''
from logging.config import fileConfig
from sqlalchemy import engine_from_config
from alembic import context

# This is the Alembic Config object
config = context.config

# Add your model's MetaData object here for 'autogenerate' support
from __main__ import Base
target_metadata = Base.metadata

def run_migrations_offline():
    url = config.get_main_option('sqlalchemy.url')
    context.configure(url=url, target_metadata=target_metadata, literal_binds=True)
    with context.begin_transaction():
        context.run_migrations()

def run_migrations_online():
    connectable = engine_from_config(
        config.get_section(config.config_ini_section),
        prefix='sqlalchemy.',
        connect_args={'user': "''' + USERNAME + '''", 'password': "''' + PASSWORD + '''", 'dsn': "''' + DSN + '''"}
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()

if context.is_offline_mode():
    run_migrations_offline()
else:
    run_migrations_online()
'''

with open('/content/alembic/env.py', 'w') as f:
    f.write(env_py)

# Write a minimal script.py.mako
script_template = '''"""${message}

Revision ID: ${up_revision}
Revises: ${down_revision | comma,n}
Create Date: ${create_date}
"""

from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

# revision identifiers, used by Alembic.
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
branch_labels = ${repr(branch_labels)}
depends_on = ${repr(depends_on)}

def upgrade():
${upgrades if upgrades else "    pass"}

def downgrade():
${downgrades if downgrades else "    pass"}
'''

with open('/content/alembic/script.py.mako', 'w') as f:
    f.write(script_template)

print('✅ Alembic initialized in /content/alembic/')

✅ Alembic initialized in /content/alembic/


In [62]:
# Generate migration from models vs current database state
command.revision(alembic_cfg, autogenerate=True, message='Initial schema')

# Show what was generated
import glob
migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
print('Generated migrations:')
for f in migration_files:
    print(f'  {f}')

Generating /content/alembic/versions/dc62f0e11bfa_initial_schema.py ...  done
Generated migrations:
  /content/alembic/versions/d67fc4608479_initial_schema.py
  /content/alembic/versions/dc62f0e11bfa_initial_schema.py


In [64]:
# Read and display the latest migration
latest = migration_files[1]
with open(latest) as f:
    content = f.read()

print(content)


"""Initial schema

Revision ID: dc62f0e11bfa
Revises: d67fc4608479
Create Date: 2026-05-12 13:49:07.882812
"""

from alembic import op
import sqlalchemy as sa


# revision identifiers, used by Alembic.
revision = 'dc62f0e11bfa'
down_revision = 'd67fc4608479'
branch_labels = None
depends_on = None

def upgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.add_column('teams', sa.Column('priority', sa.String(length=20), nullable=True))
    op.add_column('teams', sa.Column('due_date', sa.DateTime(), nullable=True))
    op.add_column('teams', sa.Column('tags', sa.String(length=500), nullable=True))
    # ### end Alembic commands ###

def downgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.drop_column('teams', 'tags')
    op.drop_column('teams', 'due_date')
    op.drop_column('teams', 'priority')
    # ### end Alembic commands ###



In [ ]:
team = Team()

In [65]:

command.upgrade(alembic_cfg, 'head')
print('✅ Migration applied! Tables created.')

✅ Migration applied! Tables created.


In [66]:
# Rollback one migration
command.downgrade(alembic_cfg, '-1')
print('✅ Downgraded by 1. New columns removed.')

✅ Downgraded by 1. New columns removed.


In [ ]:
import glob
import os

# Delete all generated migration files
migration_files = glob.glob('/content/alembic/versions/*.py')

for f in migration_files:
    os.remove(f)
    print(f"Deleted: {f}")